# Código para limpiar y transformar el precio OMIE

In [ ]:
import pandas as pd

# 1. Cargar datos
df = pd.read_csv("OMIEPrecio2020_2024.csv", sep=";")

# 2. Filtrar precios
df = df[df["CONCEPT"] == "PRICE_SP"]

# 3. Pasar a formato largo
cols_horas = [c for c in df.columns if c.startswith("H")]

df = df.melt(
    id_vars=["DATE"],
    value_vars=cols_horas,
    var_name="hour",
    value_name="price"
)

# 4. Limpiar hora
df["hour"] = df["hour"].str.replace("H", "")
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

# eliminar filas vacias
df = df.dropna(subset=["price"])

df["hour"] = df["hour"].astype(int)

# convertir a rango 0-23
df["hour"] = df["hour"] - 1

# H25 -> convertir hora 24 a 23 (cambio otoño), no importa que haya dos horas 23 en un dia luego las combinamos
df.loc[df["hour"] == 24, "hour"] = 23

# asegurar que price es numérico antes de hacer la media o interpolar
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# 5. Crear datetime
df["datetime"] = pd.to_datetime(df["DATE"]) + pd.to_timedelta(df["hour"], unit="h")

# 6. OTOÑO: eliminar duplicados con media de ambos valores repetidos
df = df.groupby("datetime", as_index=False)["price"].mean().round(2)

# 7. PRIMAVERA: rellenar horas faltantes
df = df.set_index("datetime")

# ordenar por si acaso (importante)
df = df.sort_index()

# crear rango continuo de horas
rango = pd.date_range(df.index.min(), df.index.max(), freq="h")

# reindexar -> aquí aparecen los huecos
df = df.reindex(rango)

# interpolar valores faltantes (lineal)
df["price"] = df["price"].interpolate(method="linear")

# volver a columna normal
df = df.reset_index().rename(columns={"index": "datetime"})

# 8. Obtener hora final
df["hour"] = df["datetime"].dt.hour

# 9. Ordenar
df = df.sort_values("datetime").reset_index(drop=True)

print(df.head())

# =========================
# Guardar CSV
# =========================
df.to_csv("LimpiezaOmiePrecio2020_2024.csv", sep=';', index=False)

             datetime  price  hour
0 2020-01-01 00:00:00  41.88     0
1 2020-01-01 01:00:00  38.60     1
2 2020-01-01 02:00:00  36.55     2
3 2020-01-01 03:00:00  32.32     3
4 2020-01-01 04:00:00  30.85     4


C:\Users\Jaime_Sanchez\AppData\Local\Temp\ipykernel_6984\32633521.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df = df.groupby("datetime", as_index=False)["price"].mean().round(2)


## Explicacion paso a paso

2. Filtrar 
El dataset contiene el precio iberico incluido portugal, a nosotros solo nos intersa el precio español (SP)

3. Pasar las horas de un formato por columnas a filas (1 hora = 1 col --> 1 hora = 1 fila)
itera sobre las cols del df y selecciona aquellas cuyo nombre comienza con "H"

df = df.melt(...) -> Convierte el DataFrame al formato largo (tidy data) donde cada fila representa una unica fecha y hora.

Antes:
DATE        H1    H2    H3    H4    H5  ...  H24
2024-01-01  50    52    51    49    48  ...   55
2024-01-02  53    54    52    50    51  ...   56

Despues:
DATE        hour  price
2024-01-01  H1    50
2024-01-01  H2    52
2024-01-01  H3    51
...


6. OTOÑO: eliminar duplicados por cambio a horario invierno
df.groupby("datetime", as_index=False) -> Agrupa todas las filas que tienen el mismo valor en la columna "datetime"

as_index=False -> la columna "datetime" no se convierte en índice, sigue siendo una columna normal

["price"] -> Selecciona únicamente la columna "price" para aplicar la operación de agregación

.mean() -> Calcula la media aritmética de los precios dentro de cada grupo (cada datetime duplicado)

.round(2) -> Redondea a 2 decimales, porque en el dataset vienen así todos y sino se quedarian estos con mas deciamles que el resto
